# M08 — RSA compares relationships, not matching voxel labels

<!-- paper-first -->
### Research question

**Reading:** [PM02](../../curriculum/papers/modeling.md#pm02). Review the assigned figure or result before starting the lesson.

**Question:** How can two representations be compared when their individual feature dimensions do not correspond?

Record a prediction, a source location, and one point you want this lesson to clarify. Ask your AI tutor to distinguish the paper’s evidence from its interpretation.
<!-- /paper-first -->

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

Representational similarity analysis asks whether the pattern of relationships among conditions matches a proposed representation. Two people can have different voxel-level patterns while showing similar condition geometry. An RDM, or representational dissimilarity matrix, summarizes pairwise distances between condition patterns. Conditions must have the same meaning and order across the brain, behavioral, and model RDMs.

The distance choice changes the scientific quantity. Euclidean distance is sensitive to scale; correlation distance reduces sensitivity to pattern mean and overall scale but has its own assumptions. Whitening can account for measurement-noise covariance when that covariance is estimated appropriately. A plot labeled simply similarity omits decisions necessary to interpret it. The diagonal and the repeated lower triangle also must be handled explicitly.

Noise biases ordinary squared distances upward: two noisy measurements of identical underlying patterns will usually differ. Crossvalidated distance estimates compare independent estimates of condition differences. In the simplified isotropic-noise example, we take the dot product of a condition contrast from run A with the same contrast from independent run B. With independent zero-mean noise, this estimates the squared true contrast without the same positive noise bias. A negative estimate is possible under noise; it is not a physically negative squared distance.

This notebook does not implement a complete crossnobis analysis. Crossnobis additionally involves a noise-normalized distance and careful estimation of the precision matrix, as well as suitable crossvalidation. We use an unwhitened crossvalidated squared Euclidean estimator to make the independence argument visible. Calling every crossvalidated distance crossnobis would hide a missing transformation.

RDM cells are dependent because pairs share conditions. Treating all pairwise entries as independent participants exaggerates evidence. For a permutation test of a condition model, relabel condition rows and columns coherently under a justified exchangeability scheme; do not shuffle individual edges independently. Participant-level generalization also needs participant-level estimates and uncertainty. An attractive correlation between two RDMs is not automatically a population-level inference.

The exercise first constructs an RDM and compares it with a category model. It then repeats the equal-pattern experiment many times to contrast biased ordinary distances with crossvalidated estimates. Inspect the whole distribution, including negative values. The deliberate failure reuses run A as run B, destroying independence and restoring positive bias. Ask the AI to state what is independent, where noise covariance would be estimated, and exactly which hypothesis a comparison addresses.

## Transformation contract

Condition×feature patterns → pairwise-distance RDM → upper-triangle vector → model agreement. A second independent run enables a crossvalidated contrast product; it changes estimator bias, not the number of independent participants.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from scipy.spatial.distance import pdist,squareform
from scipy.stats import spearmanr
rng=np.random.default_rng(408)
category=np.repeat([0,1],3)
pattern=category[:,None]*np.ones((6,12))+rng.normal(0,.15,(6,12))
rdm=squareform(pdist(pattern,metric='euclidean'))
model=(category[:,None]!=category[None,:]).astype(float)
upper=np.triu_indices(6,1)
r=spearmanr(rdm[upper],model[upper]).statistic
print('RDM shape / category-model Spearman:',rdm.shape,r)
assert np.allclose(rdm,rdm.T) and np.allclose(np.diag(rdm),0)
assert len(rdm[upper])==15 and r>.7


RDM shape / category-model Spearman: (6, 6) 0.8504200642707612


In [2]:
run_a=rng.normal(size=(5000,2,20));run_b=rng.normal(size=(5000,2,20))
da=run_a[:,1]-run_a[:,0];db=run_b[:,1]-run_b[:,0]
ordinary=np.sum(da*da,axis=1)
crossvalidated=np.sum(da*db,axis=1)
leaked=np.sum(da*da,axis=1)
print('Mean ordinary / independent crossvalidated / reused-run:',ordinary.mean(),crossvalidated.mean(),leaked.mean())
assert ordinary.mean()>35 and abs(crossvalidated.mean())<1
assert np.any(crossvalidated<0) and np.array_equal(ordinary,leaked)
perm=rng.permutation(6)
permuted=model[np.ix_(perm,perm)]
assert np.array_equal(permuted,permuted.T)


Mean ordinary / independent crossvalidated / reused-run: 40.215572428782124 -0.03842206372639939 40.215572428782124


## Deliberate failure and repair

Reusing the same noisy run on both sides makes the purported crossvalidated estimator an ordinary squared distance. Repair the independence of the estimates before interpreting distance. If condition order changes, permute both axes of the RDM together; shuffling edge entries independently is not a coherent relabeling.

## Your investigation

Choose and defend a distance for comparing two stimulus categories. Draw how conditions, runs, and participants enter the analysis. Explain the role a noise covariance estimate would play in crossnobis, and why the present lab does not supply it. Inspect the vendored BrainIAK RSA notebook before planning a real condition model.

## Transfer to real neuroimaging

The vendored RSA tutorial includes real fMRI data handling and a wider range of representations. Intersubject RSA compares relationships among participants, which is a different indexing scheme from condition RSA. Explicitly identify the matrix axes before transferring code.

**Primary teaching sources, pinned where hosted on GitHub:**

- [BrainIAK: RSA (vendored original)](https://github.com/brainiak/brainiak-tutorials/blob/fb62ede943d9694fe703aee0df5f43ecf5558415/tutorials/06-rsa.ipynb)
- [Dartmouth/OHBM: intersubject RSA](https://github.com/naturalistic-data-analysis/naturalistic_data_analysis/blob/88bd22741508b4d678202bde7530cee0511e283f/content/Intersubject_RSA.ipynb)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. Why can a crossvalidated estimate be negative? **Noise can make independent contrast estimates point in opposing directions.**
2. Are 15 RDM edges 15 independent subjects? **No; edges share conditions and are not participant replicates.**

### Return to the research question

Revisit [PM02](../../curriculum/papers/modeling.md#pm02) and your initial prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure or section locator. Which part of the published result remains open after this exercise?
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Include this entry in the A2 portfolio when relevant.
